# Awards SQL — LoRA Fine-tune on Colab

Trains a LoRA adapter for Llama-3.1-8B-Instruct on the federal awards text-to-SQL dataset, then evaluates it.

**Setup:**
1. Runtime → Change runtime type → **T4 GPU** (Pro) or **A100** (Pro+)
2. Run cell **(2)** to upload the 4 data files (`train.jsonl`, `val.jsonl`, `eval.jsonl`, `schema_compact.txt`)
3. Set `HF_TOKEN` in cell **(3)** — must have access to Llama 3.1 8B
4. Runtime → Run all

**Output:** at the end you'll get an `awards-sql-lora.zip` to download. Drop it back into `slm/out/` locally and we'll continue with Workers AI deploy.

## (1) Install

In [ ]:
!pip install -q -U transformers==4.46.0 peft==0.13.2 trl==0.12.0 datasets==3.1.0 accelerate==1.1.1 bitsandbytes>=0.45.0 huggingface_hub
import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('vram:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB' if torch.cuda.is_available() else '')


## (2) Upload data files
Drag-and-drop **train.jsonl**, **val.jsonl**, **eval.jsonl**, **schema_compact.txt** when prompted.

In [ ]:
from google.colab import files
import os
os.makedirs('data', exist_ok=True)
uploaded = files.upload()  # browser dialog
for fn in uploaded:
  os.rename(fn, f'data/{fn}')
!ls -la data/


## (3) HuggingFace login

In [ ]:
HF_TOKEN = 'hf_PASTE_YOUR_TOKEN_HERE'  # ← paste your hf_... token, must have Llama 3.1 access
from huggingface_hub import login
login(HF_TOKEN)


## (4) Load Llama 3.1 8B in 4-bit

In [ ]:
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
BASE = 'meta-llama/Llama-3.1-8B-Instruct'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                          bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto', torch_dtype=torch.bfloat16)
print('loaded, vram now:', torch.cuda.memory_allocated()/1e9, 'GB')


## (5) Apply LoRA (Workers AI compatible: r=16, q/k/v/o)

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## (6) Build dataset

In [ ]:
from datasets import Dataset
schema = open('data/schema_compact.txt').read()
SYSTEM = ('You are a text-to-SQL model for a SQLite database of federal contracts and grants.\n'
          'Given a question, return ONLY the SQL query. Do not include explanation or fences — just SQL ending with semicolon.\n\nSCHEMA:\n' + schema)

def to_chat(p):
  msgs = [{'role':'system','content':SYSTEM},
          {'role':'user','content':p['question']},
          {'role':'assistant','content':p['sql']}]
  return {'text': tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

train = [json.loads(l) for l in open('data/train.jsonl')]
val = [json.loads(l) for l in open('data/val.jsonl')]
train_ds = Dataset.from_list([to_chat(p) for p in train])
val_ds = Dataset.from_list([to_chat(p) for p in val])
print('train:', len(train_ds), 'val:', len(val_ds))


## (7) Train (3 epochs, ~45 min on A100, ~3 hr on T4)

In [ ]:
from trl import SFTTrainer, SFTConfig
cfg = SFTConfig(
  output_dir='out/awards-sql-lora',
  num_train_epochs=3,
  per_device_train_batch_size=2,        # T4=2, A100=8
  per_device_eval_batch_size=2,
  gradient_accumulation_steps=4,
  learning_rate=2e-4,
  lr_scheduler_type='cosine',
  warmup_ratio=0.05,
  weight_decay=0.01,
  logging_steps=10,
  save_strategy='epoch',
  eval_strategy='epoch',
  bf16=True,
  optim='paged_adamw_8bit',
  max_grad_norm=1.0,
  max_seq_length=1536,
  packing=False,
  gradient_checkpointing=False,
  dataloader_num_workers=0,
  report_to='none',
  dataset_text_field='text',
)
trainer = SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=cfg, tokenizer=tok)
trainer.train()
trainer.save_model('out/awards-sql-lora')
tok.save_pretrained('out/awards-sql-lora')
print('saved')


## (8) Eval the trained adapter on eval.jsonl

In [ ]:
import re, sqlite3, json
# Need a SQLite copy to validate execution. We'll quickly rebuild a minimal one
# from schema only (just for parse-checking; real exec accuracy is computed locally).
# Here we report PARSE accuracy + the model's predicted SQL so you can review.
model.eval()
eval_set = [json.loads(l) for l in open('data/eval.jsonl')]
results = []
def extract(t):
  t = t.strip()
  m = re.search(r'```(?:sql)?\s*(.*?)```', t, re.DOTALL|re.I)
  if m: t = m.group(1)
  m = re.match(r'\s*(.*?;)', t, re.DOTALL)
  if m: t = m.group(1)
  return t.strip()

for ex in eval_set:
  msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':ex['question']}]
  text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
  ids = tok(text, return_tensors='pt').to(model.device)
  with torch.no_grad():
    out = model.generate(**ids, max_new_tokens=400, do_sample=False, pad_token_id=tok.eos_token_id)
  gen = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
  pred = extract(gen)
  results.append({'question': ex['question'], 'gold_sql': ex['sql'], 'pred_sql': pred, 'category': ex['category']})

with open('out/awards-sql-lora/eval_predictions.json', 'w') as f:
  json.dump(results, f, indent=2)
for r in results[:5]:
  print('Q:', r['question'])
  print('  GOLD:', r['gold_sql'][:120])
  print('  PRED:', r['pred_sql'][:120])
  print()


## (9) Zip + download adapter

In [ ]:
!cd out && zip -r awards-sql-lora.zip awards-sql-lora
!ls -la out/awards-sql-lora.zip
from google.colab import files
files.download('out/awards-sql-lora.zip')
